In [ ]:
from SPARQLWrapper import SPARQLWrapper
from SPARQLWrapper import JSON, CSV
from SPARQLWrapper import TURTLE
from pathlib import Path
import os
import pandas as pd
import geopandas as gpd
import pandas as pd
import shapely.wkt
import io

In [ ]:
endpoint = SPARQLWrapper("http://anubis:7200/repositories/tests")
endpoint.setReturnFormat(JSON)

excluded_uris = """
        <http://rdf.geohistoricaldata.org/id/museeCarnavalet/ephemeres/agent/imprimerieflevee>,
        <http://rdf.geohistoricaldata.org/id/museeCarnavalet/ephemeres/agent/typographiepanckoucke>,
        <http://rdf.geohistoricaldata.org/id/museeCarnavalet/ephemeres/agent/imprimerieadindcbourgerie>,
        <http://rdf.geohistoricaldata.org/id/museeCarnavalet/ephemeres/agent/reullierj>,
        <http://rdf.geohistoricaldata.org/id/museeCarnavalet/ephemeres/agent/clavela>,
        <http://rdf.geohistoricaldata.org/id/museeCarnavalet/ephemeres/agent/henrysicard>,
        <http://rdf.geohistoricaldata.org/id/museeCarnavalet/ephemeres/agent/delanchyimprimerie>,
        <http://rdf.geohistoricaldata.org/id/museeCarnavalet/ephemeres/agent/imprimerieadavy>,
        <http://rdf.geohistoricaldata.org/id/museeCarnavalet/ephemeres/agent/imprimeriehenon>,
        <http://rdf.geohistoricaldata.org/id/museeCarnavalet/ephemeres/agent/imprimerieedupre>,
        <http://rdf.geohistoricaldata.org/id/museeCarnavalet/ephemeres/agent/courmontfreres>,
        <http://rdf.geohistoricaldata.org/id/museeCarnavalet/ephemeres/agent/panckouckecharleslouisfleury>,
        <http://rdf.geohistoricaldata.org/id/museeCarnavalet/ephemeres/agent/imprimeriegrandremyethenon>,
        <http://rdf.geohistoricaldata.org/id/museeCarnavalet/ephemeres/agent/imprimerierthomascie>,
        <http://rdf.geohistoricaldata.org/id/museeCarnavalet/ephemeres/agent/imprimerielmichelcie>,
        <http://rdf.geohistoricaldata.org/id/museeCarnavalet/ephemeres/agent/imprimeriegdemalherbeetcie>,
        <http://rdf.geohistoricaldata.org/id/museeCarnavalet/ephemeres/agent/imprimeriebourgerieetcie>,
        <http://rdf.geohistoricaldata.org/id/museeCarnavalet/ephemeres/agent/imprimeriebelfondcie>,
        <http://rdf.geohistoricaldata.org/id/museeCarnavalet/ephemeres/agent/imprimerieajanniotcie>,
        <http://rdf.geohistoricaldata.org/id/museeCarnavalet/ephemeres/agent/heliotypiebuirettecie>,
        <http://rdf.geohistoricaldata.org/id/museeCarnavalet/ephemeres/agent/cussetetcieimprimerie>

    """

query_date_intervalle = f"""
        PREFIX adb: <http://data.soduco.fr/def/annuaire#>
        PREFIX rico: <https://www.ica.org/standards/RiC/ontology#>
        PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
        PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

        SELECT DISTINCT ?uriAgent ?nomagent ?typeAgent (MIN(?debut) AS ?dateDebutMin) (MAX(?fin) AS ?dateFinMax) ?adresse ?geom
        WHERE {{
            ?rel a rico:CreationRelation ;
                rico:withCreationRole ?role ;
                rico:relationHasTarget ?uriAgent ;
                rico:relationHasSource ?s .
                
            ?uriAgent rico:name ?nomagent .
            ?role skos:prefLabel ?typeAgent .
            ?cluster skos:exactMatch ?uriAgent .
            ?cluster adb:address ?adresse .
            ?cluster adb:hasAddressGeometry ?geom .
            
            ?s rico:hasCreationDate ?date .
            ?date rico:beginningDate ?debut ;
                rico:endDate ?fin .
            FILTER(!CONTAINS(LCASE(STR(?nomagent)), "anonyme"))
            FILTER(?uriAgent NOT IN ({excluded_uris}))
        }}
        GROUP BY ?uriAgent ?nomagent ?typeAgent ?adresse ?geom
    """

In [74]:
query_liens = f"""
        PREFIX rico: <https://www.ica.org/standards/RiC/ontology#>

        SELECT ?agent1 ?agent2 (COUNT(DISTINCT ?s) AS ?weight)
        WHERE {{
            ?rel1 a rico:CreationRelation ; rico:relationHasSource ?s ; rico:relationHasTarget ?agent1 .
            ?rel2 a rico:CreationRelation ; rico:relationHasSource ?s ; rico:relationHasTarget ?agent2 .
            FILTER(STR(?agent1) < STR(?agent2))
            FILTER(?agent1 NOT IN ({excluded_uris}))
            FILTER(?agent2 NOT IN ({excluded_uris}))
        }}
        GROUP BY ?agent1 ?agent2
    """

In [ ]:
endpoint.setQuery(query_date_intervalle)
endpoint.setReturnFormat(CSV)
resultats_csv = endpoint.queryAndConvert()
df = pd.read_csv(io.BytesIO(resultats_csv))

df.head()

,uriAgent,nomagent,typeAgent,dateDebutMin,dateFinMax,adresse,geom
0,http://rdf.geohistoricaldata.org/id/museeCarna...,"Hardy, Dudley",Dessinateur,1890,1900,33 r. Fortu-ny,POINT(2.307839 48.882934)
1,http://rdf.geohistoricaldata.org/id/museeCarna...,Joly (Éditeur de musique),Imprimeur,1880,1900,14 Renard,POINT(2.351709 48.857991)
2,http://rdf.geohistoricaldata.org/id/museeCarna...,Imprimerie A. Thomas,Imprimeur,1880,1900,r. des Fourneaux,POINT(2.305183 48.829954)
3,http://rdf.geohistoricaldata.org/id/museeCarna...,Imprimerie Dubert fils,Imprimeur,1886,1886,13 rue de la Chapelle,POINT(2.359102 48.88545)
4,http://rdf.geohistoricaldata.org/id/museeCarna...,"Berni, G.",Dessinateur,1880,1900,9 Charlot,POINT(2.360428 48.861147)


In [ ]:
endpoint.setQuery(query_liens)
endpoint.setReturnFormat(CSV)
resultats_csv_2 = endpoint.queryAndConvert()
df_2 = pd.read_csv(io.BytesIO(resultats_csv_2))

df_2.head()

,agent1,agent2,weight
0,http://rdf.geohistoricaldata.org/id/museeCarna...,http://rdf.geohistoricaldata.org/id/museeCarna...,12
1,http://rdf.geohistoricaldata.org/id/museeCarna...,http://rdf.geohistoricaldata.org/id/museeCarna...,6
2,http://rdf.geohistoricaldata.org/id/museeCarna...,http://rdf.geohistoricaldata.org/id/museeCarna...,3
3,http://rdf.geohistoricaldata.org/id/museeCarna...,http://rdf.geohistoricaldata.org/id/museeCarna...,3
4,http://rdf.geohistoricaldata.org/id/museeCarna...,http://rdf.geohistoricaldata.org/id/museeCarna...,2


In [ ]:
df["dateDebutMin"] = pd.to_numeric(df["dateDebutMin"], errors="coerce")
df["dateFinMax"] = pd.to_numeric(df["dateFinMax"], errors="coerce")

In [78]:
output_dir = Path("gpkg_pour_carto")  
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
agents_presents = set(df['uriAgent'])
df_filtered_links = df_2[
    df_2['agent1'].isin(agents_presents) & 
    df_2['agent2'].isin(agents_presents)
]

links_combined = pd.concat([
    df_filtered_links[['agent1', 'weight']].rename(columns={'agent1': 'uriAgent'}),
    df_filtered_links[['agent2', 'weight']].rename(columns={'agent2': 'uriAgent'})
])

degree_df = links_combined.groupby('uriAgent').agg(
    degree=('weight', 'count'),          # Nombre d'agents différents avec qui il a collaboré
    weighted_degree=('weight', 'sum')    # Volume cumulé des affiches/sources de ces collaborations
).reset_index()

df = df.merge(degree_df, on='uriAgent', how='left')
df['degree'] = df['degree'].fillna(0).astype(int)
df['weighted_degree'] = df['weighted_degree'].fillna(0).astype(int)

df["geometry"] = df["geom"].apply(shapely.wkt.loads)
gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")
gdf = gdf.drop(columns=["geom"])

gdf.to_file(
    output_dir/"tous_les_agents.gpkg",
    driver="GPKG",
    layer="agents",
)

In [ ]:
from shapely.geometry import LineString
gdf_l93 = gdf.to_crs("EPSG:2154")

gdf_nodes_subset = gdf_l93[['uriAgent', 'geometry']].drop_duplicates(subset=['uriAgent'])

df_links = df_2.merge(gdf_nodes_subset, left_on='agent1', right_on='uriAgent', how='inner')
df_links = df_links.rename(columns={'geometry': 'geom1'}).drop(columns=['uriAgent'])

df_links = df_links.merge(gdf_nodes_subset, left_on='agent2', right_on='uriAgent', how='inner')
df_links = df_links.rename(columns={'geometry': 'geom2'}).drop(columns=['uriAgent'])

line_records = []
for _, row in df_links.iterrows():
    p1 = row['geom1']
    p2 = row['geom2']
    
    if p1 is not None and p2 is not None and not p1.is_empty and not p2.is_empty:
        line_records.append({
            'agent1': row['agent1'],
            'agent2': row['agent2'],
            'weight': row['weight'],
            'geometry': LineString([p1, p2])
        })

gpkg_path = output_dir / "tous_les_agents.gpkg"
gdf_lines = gpd.GeoDataFrame(line_records, crs="EPSG:2154")
gdf_lines.to_file(gpkg_path, layer="liens_reseau", driver="GPKG")

print(f"Couche 'liens_reseau' ajoutée avec succès avec {len(gdf_lines)} lignes dans : {gpkg_path}")

Couche 'liens_reseau' ajoutée avec succès avec 192 lignes dans : gpkg_pour_carto/tous_les_agents.gpkg


In [ ]:
gdf_imp_dess = gdf[gdf["typeAgent"].isin (["Imprimeur", "Dessinateur"])]

gdf_imp_dess.to_file(
    "imprimeurs_dessinateurs.gpkg",
    driver="GPKG",
    layer="imprimeurs_dessinateurs",
)


In [ ]:
gdf_imp_dess_1845_1853 = gdf_imp_dess [(gdf_imp_dess["dateDebutMin"] <= 1853) & (gdf["dateFinMax"] >= 1845)]
gdf_imp_dess_1845_1853.to_file(
    output_dir/"imprimeurs_dessinateurs_1845-1853.gpkg",
    driver="GPKG",
    layer="imprimeurs_dessinateurs",
)

/home/vmerilan/Documents/VMerilan/scripts/algo_traitement_v1/v1_exec/.venv/lib/python3.13/site-packages/geopandas/geodataframe.py:1891: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  result = super().__getitem__(key)


In [ ]:
gdf_imp_dess_1854_1872 = gdf_imp_dess [(gdf_imp_dess["dateDebutMin"] <= 1872) & (gdf["dateFinMax"] >= 1854)]
gdf_imp_dess_1854_1872.to_file(
    output_dir/"imprimeurs_dessinateurs_1854-1872.gpkg",
    driver="GPKG",
    layer="imprimeurs_dessinateurs",
)

/home/vmerilan/Documents/VMerilan/scripts/algo_traitement_v1/v1_exec/.venv/lib/python3.13/site-packages/geopandas/geodataframe.py:1891: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  result = super().__getitem__(key)


In [ ]:
gdf_imp_dess_1873_1913 = gdf_imp_dess [(gdf_imp_dess["dateDebutMin"] <= 1913) & (gdf["dateFinMax"] >= 1873)]
gdf_imp_dess_1873_1913.to_file(
    output_dir/"imprimeurs_dessinateurs_1873-1913.gpkg",
    driver="GPKG",
    layer="imprimeurs_dessinateurs",
)

/home/vmerilan/Documents/VMerilan/scripts/algo_traitement_v1/v1_exec/.venv/lib/python3.13/site-packages/geopandas/geodataframe.py:1891: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  result = super().__getitem__(key)


In [ ]:
gdf_imp_dess_1911_1919 = gdf_imp_dess [(gdf_imp_dess["dateDebutMin"] <= 1919) & (gdf["dateFinMax"] >= 1911)]
gdf_imp_dess_1911_1919.to_file(
    output_dir/"imprimeurs_dessinateurs_1911-1919.gpkg",
    driver="GPKG",
    layer="imprimeurs_dessinateurs",
)

/home/vmerilan/Documents/VMerilan/scripts/algo_traitement_v1/v1_exec/.venv/lib/python3.13/site-packages/geopandas/geodataframe.py:1891: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  result = super().__getitem__(key)


In [ ]:
gdf_imp = gdf[gdf["typeAgent"] == "Imprimeur"]
gdf_imp.to_file(
    output_dir/"imprimeurs.gpkg",
    driver="GPKG",
    layer="imprimeurs_dessinateurs",
)

In [ ]:
gdf_dess = gdf[gdf["typeAgent"] == "Dessinateur"]
gdf_dess.to_file(
    output_dir/"dessinateurs.gpkg",
    driver="GPKG",
    layer="imprimeurs_dessinateurs",
)